# 07 · Every number in the paper, from code

`src/paper_numbers.py` recomputes each number the manuscript quotes from the pipeline outputs and records the table
or code it comes from (`outputs/tables/paper_numbers.csv`). This notebook then checks the manuscript text in both
directions:

1. every number in the text is in the registry (otherwise it has no code behind it);
2. every registry value appears in the text as printed (otherwise the text is out of date).

The manuscript text is read from `manuscript/manuscript.md` (kept private until publication). Without it, the registry is still written.
Run after notebook 06.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import paper_numbers as pn
pd.set_option("display.max_colwidth", 120)

reg = pn.registry(ROOT)
reg.to_csv(ROOT / "outputs" / "tables" / "paper_numbers.csv", index=False)
print(f"{len(reg)} numbers recomputed; {reg.source.str.startswith('[').sum()} are cited external facts.")
reg.groupby("section").size().rename("numbers")

203 numbers recomputed; 4 are cited external facts.


section
Abstract           1
Discussion 4.2     1
External           4
Methods 2.1        8
Methods 2.3        9
Methods 2.4        1
Methods 2.5        1
Results 3.1       17
Results 3.2       15
Results 3.3       14
Results 3.4       32
Results 3.5       17
Table 1           16
Table 2           50
Table 3           17
Name: numbers, dtype: int64

In [2]:
MS = [p for p in [ROOT / "manuscript" / "manuscript.md"] if p.exists()]
if MS:
    ms = MS[-1]
    text = ms.read_text()
    unmatched = pn.check_manuscript(ms, reg)
    reg["in_manuscript"] = [str(v) in text for v in reg["value"]]
    print(f"{ms.name}: {len(unmatched)} number(s) in the text without code behind them; "
          f"{(~reg.in_manuscript).sum()} registry value(s) not printed as computed.")
    display(unmatched)
    display(reg.loc[~reg.in_manuscript, ["section", "claim", "value", "source"]])
else:
    print("No manuscript/manuscript.md found: registry written, text not checked.")

manuscript.md: 0 number(s) in the text without code behind them; 0 registry value(s) not printed as computed.


,number,context


,section,claim,value,source
